<a href="https://colab.research.google.com/github/emdt-cyber/test-capstone/blob/main/test_sudhra_hua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**README.md**

In [94]:
!git clone https://github.com/emdt-cyber/delhivery-delay-pipeline.git

Cloning into 'delhivery-delay-pipeline'...
fatal: could not read Username for 'https://github.com': No such device or address


In [95]:
%cd delhivery-delay-pipeline

[Errno 2] No such file or directory: 'delhivery-delay-pipeline'
/content


In [96]:
!mkdir -p data src models outputs

In [97]:
!pwd
!ls

/content
data	 package_model.py      part_c_classification.py  reports
models	 part_a_clean_eda.py   part_d_segments.py	 sample_data
outputs  part_b_regression.py  predict_delay.py		 src


In [98]:
!git clone https://github.com/emdt-cyber/delhivery-delay-pipeline.git

Cloning into 'delhivery-delay-pipeline'...
fatal: could not read Username for 'https://github.com': No such device or address


In [99]:
%cd delhivery-delay-pipeline

[Errno 2] No such file or directory: 'delhivery-delay-pipeline'
/content


**requirements.txt**

In [100]:
import pandas as pd
import numpy as np
import sklearn
import joblib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1
Joblib: 1.6.0


**generate_data.py**

In [101]:
"""Synthetic shipment data for the Delay Intelligence capstone.
Run once:  python generate_data.py   (writes data/hubs.csv, data/raw_shipments.csv, data/new_shipments.csv)
Do not edit any constant: every number in the brief was measured on this exact output."""
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))

24 1224 48


**shipment_prep.py**

In [102]:
"""Shared shipment cleaning and feature engineering for the capstone."""
import numpy as np
import pandas as pd

PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}

FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value",
    "n_stops", "rain_flag", "sale_event_flag", "is_cod", "is_weekend",
    "dispatch_hour", "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no",
    "sticker_serial_mod", "shelf_slot_no",
]

NOISE_COLUMNS = [
    "label_print_batch", "scanner_build_no", "tape_roll_no",
    "sticker_serial_mod", "shelf_slot_no",
]


def clean_shipments(raw, medians=None):
    """Return (clean_df, medians) after applying the specified cleaning steps."""
    df = raw.drop_duplicates().copy()
    df["service_type"] = df["service_type"].astype(str).str.strip().str.title()
    df["payment_mode"] = df["payment_mode"].fillna("Unknown")
    df["declared_value_inr"] = (
        df["declared_value"].astype(str)
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
        .astype(float)
    )
    df = df.drop(columns=["declared_value"])
    df["dispatch_time"] = pd.to_datetime(df["dispatch_time"])
    df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    if medians is None:
        medians = {
            "distance_km": float(df["distance_km"].median()),
            "parcel_weight_kg": float(df["parcel_weight_kg"].median()),
            "declared_value_inr": float(df["declared_value_inr"].median()),
        }
    else:
        required = {"distance_km", "parcel_weight_kg", "declared_value_inr"}
        if set(medians) != required:
            raise ValueError(f"medians must contain exactly these keys: {sorted(required)}")
        medians = {k: float(v) for k, v in medians.items()}

    for column in ("distance_km", "parcel_weight_kg", "declared_value_inr"):
        df[column] = df[column].fillna(medians[column])
    return df, medians


def build_features(clean, hubs):
    """Attach hub information and create features; does not use delivery_hours."""
    df = clean.merge(hubs, on="hub_id", how="left", validate="many_to_one")
    if df[["hub_city", "region", "hub_tier", "hub_daily_capacity"]].isna().any().any():
        raise ValueError("One or more shipment hub_ids did not match hubs.csv")
    df["promised_hours"] = df["service_type"].map(PROMISED_HOURS).astype(int)
    df["is_cod"] = (df["payment_mode"] == "COD").astype(int)
    df["is_weekend"] = (df["dispatch_time"].dt.dayofweek >= 5).astype(int)
    df["dispatch_hour"] = df["dispatch_time"].dt.hour.astype(int)
    df["log_declared_value"] = np.log(df["declared_value_inr"].astype(float))
    return df

In [103]:
import os

print("Current directory:", os.getcwd())
print("Files:", os.listdir())

Current directory: /content
Files: ['.config', 'part_d_segments.py', 'part_a_clean_eda.py', 'part_b_regression.py', 'package_model.py', 'data', 'reports', 'src', 'outputs', 'models', 'predict_delay.py', 'part_c_classification.py', 'sample_data']


In [104]:
import os

os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Create package initialization file
with open("src/__init__.py", "w") as f:
    f.write("")

print("Project folders created successfully!")

Project folders created successfully!


In [105]:
%%writefile src/shipment_prep.py

import numpy as np
import pandas as pd

FEATURES = [
    "distance_km",
    "parcel_weight_kg",
    "declared_value_inr",
    "payment_mode",
    "service_type",
    "hub_tier"
]


def clean_shipments(raw):

    df = raw.drop_duplicates().copy()

    # Clean column text
    text_columns = ["service_type", "payment_mode"]

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip().str.title()

    # Convert numerical columns
    numeric_columns = [
        "distance_km",
        "parcel_weight_kg",
        "declared_value"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Replace invalid sentinel weight values
    df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    # Calculate medians
    medians = {
        "distance_km": df["distance_km"].median(),
        "parcel_weight_kg": df["parcel_weight_kg"].median(),
        "declared_value_inr": df["declared_value"].median()
    }

    # Fill missing numeric values
    df["distance_km"] = df["distance_km"].fillna(
        medians["distance_km"]
    )

    df["parcel_weight_kg"] = df["parcel_weight_kg"].fillna(
        medians["parcel_weight_kg"]
    )

    df["declared_value"] = df["declared_value"].fillna(
        medians["declared_value_inr"]
    )

    # Rename declared value
    df = df.rename(
        columns={"declared_value": "declared_value_inr"}
    )

    # Fill missing categorical values
    for col in ["payment_mode", "service_type"]:
        df[col] = df[col].fillna("Unknown")

    return df, medians


def build_features(clean, hubs):

    df = clean.copy()

    hubs = hubs.copy()

    # Standardize hub identifiers
    for col in ["hub_id"]:
        if col in df.columns and col in hubs.columns:
            df[col] = df[col].astype(str).str.strip()
            hubs[col] = hubs[col].astype(str).str.strip()

    # Merge hub information
    df = df.merge(
        hubs,
        on="hub_id",
        how="left",
        validate="many_to_one"
    )

    # Check required hub information
    if "hub_tier" not in df.columns:
        raise ValueError(
            "hub_tier column missing after merging hubs.csv"
        )

    df["hub_tier"] = (
        df["hub_tier"]
        .astype("string")
        .str.strip()
        .str.title()
        .fillna("Unknown")
    )

    # Check modelling columns
    required_columns = FEATURES + [
        "delivery_hours",
        "promised_hours"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    return df

Overwriting src/shipment_prep.py


In [106]:
import os

required_files = [
    "src/__init__.py",
    "src/shipment_prep.py",
]

for file in required_files:
    print(
        file,
        "FOUND" if os.path.exists(file) else "MISSING"
    )

src/__init__.py FOUND
src/shipment_prep.py FOUND


In [107]:
import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import src.shipment_prep

from src.shipment_prep import (
    clean_shipments,
    build_features,
    FEATURES
)

print("Import successful!")
print("Features:", FEATURES)

Import successful!
Features: ['distance_km', 'parcel_weight_kg', 'declared_value_inr', 'payment_mode', 'service_type', 'hub_tier']


In [108]:
import os

for file in [
    "data/raw_shipments.csv",
    "data/hubs.csv"
]:
    print(file, "FOUND" if os.path.exists(file) else "MISSING")

data/raw_shipments.csv FOUND
data/hubs.csv FOUND


In [109]:
from src.shipment_prep import clean_shipments, build_features, FEATURES

In [110]:
import os

files = [
    "data/modelling_table.csv",
    "outputs/part_a_checks.csv",
    "outputs/part_a_delay_by_group.csv",
    "outputs/part_a_correlations.csv"
]

for file in files:
    print(file, "FOUND" if os.path.exists(file) else "MISSING")

data/modelling_table.csv FOUND
outputs/part_a_checks.csv FOUND
outputs/part_a_delay_by_group.csv FOUND
outputs/part_a_correlations.csv FOUND


In [111]:
import pandas as pd

raw = pd.read_csv("data/raw_shipments.csv")

print("RAW SHIPMENT COLUMNS:")
print(raw.columns.tolist())

print("\nFIRST 5 ROWS:")
display(raw.head())

RAW SHIPMENT COLUMNS:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours']

FIRST 5 ROWS:


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [112]:
possible_columns = [
    col for col in raw.columns
    if any(word in col.lower() for word in [
        "promised",
        "expected",
        "sla",
        "delivery",
        "hour",
        "time"
    ])
]

print("Possible delivery-related columns:")
print(possible_columns)

Possible delivery-related columns:
['dispatch_time', 'delivery_hours']


In [113]:
import pandas as pd

raw = pd.read_csv("data/raw_shipments.csv")

print("Total columns:", len(raw.columns))

for i, col in enumerate(raw.columns, 1):
    print(i, ":", col)

Total columns: 17
1 : shipment_id
2 : hub_id
3 : service_type
4 : payment_mode
5 : parcel_weight_kg
6 : distance_km
7 : declared_value
8 : n_stops
9 : rain_flag
10 : sale_event_flag
11 : dispatch_time
12 : label_print_batch
13 : scanner_build_no
14 : tape_roll_no
15 : sticker_serial_mod
16 : shelf_slot_no
17 : delivery_hours


In [114]:
pd.set_option("display.max_columns", None)

display(raw.head(10))

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32
5,SHP10006,HUB21,Standard,Prepaid,1.54,589.5,₹767,62,0,0,2026-09-12 12:38,3296,25,241,0.748,93,54.49
6,SHP10007,HUB04,Express,Prepaid,4.82,189.5,"₹2,698",47,0,0,2026-09-05 08:48,5359,2,52,0.368,76,20.83
7,SHP10008,HUB17,STANDARD,COD,1.82,328.6,"₹2,411",76,0,0,2026-08-20 13:16,2454,25,262,0.805,282,53.98
8,SHP10009,HUB16,Standard,Prepaid,1.92,367.2,₹679,59,0,1,2026-09-21 10:40,1775,3,238,0.025,40,50.30
9,SHP10010,HUB01,Standard,COD,5.42,178.2,₹893,61,0,0,2026-08-19 09:00,8951,8,210,0.222,12,33.08


In [115]:
print(raw.dtypes)

shipment_id            object
hub_id                 object
service_type           object
payment_mode           object
parcel_weight_kg      float64
distance_km           float64
declared_value         object
n_stops                 int64
rain_flag               int64
sale_event_flag         int64
dispatch_time          object
label_print_batch       int64
scanner_build_no        int64
tape_roll_no            int64
sticker_serial_mod    float64
shelf_slot_no           int64
delivery_hours        float64
dtype: object


In [116]:
for col in ["dispatch_time", "delivery_hours"]:
    if col in raw.columns:
        print("\nCOLUMN:", col)
        print(raw[col].head(10))
        print("Missing values:", raw[col].isna().sum())
        print("Unique values:", raw[col].nunique())


COLUMN: dispatch_time
0    2026-08-23 19:40
1    2026-08-16 07:42
2    2026-08-01 12:15
3    2026-07-20 17:44
4    2026-07-19 17:31
5    2026-09-12 12:38
6    2026-09-05 08:48
7    2026-08-20 13:16
8    2026-09-21 10:40
9    2026-08-19 09:00
Name: dispatch_time, dtype: object
Missing values: 0
Unique values: 1192

COLUMN: delivery_hours
0    34.79
1    58.56
2    76.56
3    41.18
4    47.32
5    54.49
6    20.83
7    53.98
8    50.30
9    33.08
Name: delivery_hours, dtype: float64
Missing values: 0
Unique values: 1093


In [117]:
import pandas as pd

raw = pd.read_csv("data/raw_shipments.csv")

print("ALL COLUMN NAMES:")
print(raw.columns.tolist())

print("\nDATASET INFORMATION:")
raw.info()

ALL COLUMN NAMES:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours']

DATASET INFORMATION:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1224 entries, 0 to 1223
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   shipment_id         1224 non-null   object 
 1   hub_id              1224 non-null   object 
 2   service_type        1224 non-null   object 
 3   payment_mode        1206 non-null   object 
 4   parcel_weight_kg    1199 non-null   float64
 5   distance_km         1188 non-null   float64
 6   declared_value      1212 non-null   object 
 7   n_stops             1224 non-null   int64  
 8   rain_flag           1224 non-null   int64  
 9   sale_event_flag     1224

In [118]:
print("Delivery hours statistics:")

print(raw["delivery_hours"].describe())

print("\nDelivery duration ranges:")

print("Minimum:", raw["delivery_hours"].min())
print("Maximum:", raw["delivery_hours"].max())
print("Mean:", raw["delivery_hours"].mean())
print("Median:", raw["delivery_hours"].median())

Delivery hours statistics:
count    1224.000000
mean       56.953301
std        19.146707
min        16.360000
25%        43.267500
50%        54.435000
75%        72.465000
max       117.180000
Name: delivery_hours, dtype: float64

Delivery duration ranges:
Minimum: 16.36
Maximum: 117.18
Mean: 56.953300653594766
Median: 54.435


In [119]:
hubs = pd.read_csv("data/hubs.csv")

print("HUB COLUMNS:")
print(hubs.columns.tolist())

display(hubs.head(10))

HUB COLUMNS:
['hub_id', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity']


,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300
5,HUB06,Pune,West,1,1250
6,HUB07,Kolkata,East,1,1200
7,HUB08,Ahmedabad,West,1,1150
8,HUB09,Jaipur,North,2,950
9,HUB10,Lucknow,North,2,900


In [120]:
%%writefile src/shipment_prep.py

import numpy as np
import pandas as pd

# SLA mapping from the Part A project implementation
PROMISED_HOURS = {
    "Express": 10,
    "Standard": 28,
    "Economy": 58
}

FEATURES = [
    "distance_km",
    "parcel_weight_kg",
    "declared_value_inr",
    "payment_mode",
    "service_type",
    "hub_tier"
]


def clean_shipments(raw, medians=None):

    df = raw.drop_duplicates().copy()

    df["service_type"] = (
        df["service_type"].astype(str).str.strip().str.title()
    )

    df["payment_mode"] = df["payment_mode"].fillna("Unknown")

    numeric_columns = [
        "distance_km",
        "parcel_weight_kg",
        "declared_value"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    df["declared_value_inr"] = df["declared_value"]
    df = df.drop(columns=["declared_value"])

    if medians is None:
        medians = {
            "distance_km": float(df["distance_km"].median()),
            "parcel_weight_kg": float(df["parcel_weight_kg"].median()),
            "declared_value_inr": float(df["declared_value_inr"].median())
        }

    for col in medians:
        df[col] = df[col].fillna(medians[col])

    df["payment_mode"] = df["payment_mode"].fillna("Unknown")

    return df, medians


def build_features(clean, hubs):

    df = clean.copy()

    df["service_type"] = (
        df["service_type"].astype(str).str.strip().str.title()
    )

    df = df.merge(
        hubs,
        on="hub_id",
        how="left",
        validate="many_to_one"
    )

    # Generate promised delivery hours
    df["promised_hours"] = df["service_type"].map(PROMISED_HOURS)

    # Validate service type mapping
    if df["promised_hours"].isna().any():
        unknown = df.loc[
            df["promised_hours"].isna(),
            "service_type"
        ].unique()

        raise ValueError(f"Unknown service types: {unknown}")

    df["hub_tier"] = (
        df["hub_tier"].astype("string").str.strip().str.title()
    )

    required_columns = FEATURES + [
        "delivery_hours",
        "promised_hours"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return df

Overwriting src/shipment_prep.py


In [121]:
import importlib
import src.shipment_prep

importlib.reload(src.shipment_prep)

from src.shipment_prep import (
    clean_shipments,
    build_features,
    FEATURES
)

print("Shipment preparation module updated successfully!")

Shipment preparation module updated successfully!


In [122]:
import pandas as pd

raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

clean, medians = clean_shipments(raw)

full = build_features(clean, hubs)

print("Promised hours successfully generated!")

display(
    full[
        ["service_type", "delivery_hours", "promised_hours"]
    ].head(10)
)

Promised hours successfully generated!


,service_type,delivery_hours,promised_hours
0,Express,34.79,10
1,Standard,58.56,28
2,Economy,76.56,58
3,Standard,41.18,28
4,Standard,47.32,28
5,Standard,54.49,28
6,Express,20.83,10
7,Standard,53.98,28
8,Standard,50.30,28
9,Standard,33.08,28


In [123]:
full["delivery_hours"] = pd.to_numeric(
    full["delivery_hours"],
    errors="raise"
)

full["is_delayed"] = (
    full["delivery_hours"] > full["promised_hours"]
).astype(int)

print("Delayed shipment calculation successful!")

print("\nDelay distribution:")
print(full["is_delayed"].value_counts())

print(
    "\nDelay rate:",
    round(full["is_delayed"].mean() * 100, 2),
    "%"
)

Delayed shipment calculation successful!

Delay distribution:
is_delayed
1    1200
Name: count, dtype: int64

Delay rate: 100.0 %


In [124]:
import os

os.makedirs("data", exist_ok=True)

full.to_csv(
    "data/modelling_table.csv",
    index=False
)

print("Modelling table saved successfully!")

Modelling table saved successfully!


In [125]:
check = pd.read_csv("data/modelling_table.csv")

print("Total shipments:", len(check))

print("Promised hours exists:",
      "promised_hours" in check.columns)

print("Delay target exists:",
      "is_delayed" in check.columns)

print("Missing promised hours:",
      check["promised_hours"].isna().sum())

print("Missing delay labels:",
      check["is_delayed"].isna().sum())

display(check.head())

Total shipments: 1200
Promised hours exists: True
Delay target exists: True
Missing promised hours: 0
Missing delay labels: 0


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr,hub_city,region,hub_tier,hub_daily_capacity,promised_hours,is_delayed
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79,NaN,Lucknow,North,2,900,10,1
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56,NaN,Lucknow,North,2,900,28,1
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56,NaN,Bhubaneswar,East,3,390,58,1
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18,NaN,Bengaluru,South,1,1450,28,1
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32,NaN,Coimbatore,South,2,740,28,1


**part_a_clean_eda.py**

In [151]:
"""Part A: clean shipments, build the modelling table and produce evidence CSVs."""
import os
import numpy as np
import pandas as pd
from src.shipment_prep import clean_shipments, build_features, FEATURES

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

# Counts before cleaning, with duplicate removal applied as required.
raw_rows = len(raw)
dedup = raw.drop_duplicates().copy()
rows_after_dedup = len(dedup)
duplicate_rows_removed = raw_rows - rows_after_dedup
missing_distance_before = int(dedup["distance_km"].isna().sum())
missing_parcel_weight_before = int(dedup["parcel_weight_kg"].isna().sum())
missing_declared_value_before = int(dedup["declared_value"].isna().sum())
missing_payment_mode_before = int(dedup["payment_mode"].isna().sum())
sentinel_weight_rows = int((dedup["parcel_weight_kg"] > 100).sum())
service_type_spellings_before = int(dedup["service_type"].nunique())
service_type_spellings_after = int(dedup["service_type"].astype(str).str.strip().str.title().nunique())

clean, medians = clean_shipments(raw)
full = build_features(clean, hubs)
full["delivery_hours"] = pd.to_numeric(full["delivery_hours"], errors="raise")
full["is_delayed"] = (full["delivery_hours"] > full["promised_hours"]).astype(int)

# Confirm the left merge did not add or lose shipment rows.
rows_after_hub_merge = len(full)
full.to_csv("data/modelling_table.csv", index=False)

# Distribution and outlier checks. Bounds use unrounded quartiles.
q1 = float(full["delivery_hours"].quantile(0.25))
q3 = float(full["delivery_hours"].quantile(0.75))
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
hours_outlier_count = int(((full["delivery_hours"] < lower_bound) | (full["delivery_hours"] > upper_bound)).sum())

nulls_in_features_and_target = int(full[FEATURES + ["delivery_hours", "is_delayed"]].isna().sum().sum())
filled_log = np.log(full["declared_value_inr"].astype(float))
checks = [
    ("raw_rows", raw_rows),
    ("duplicate_rows_removed", duplicate_rows_removed),
    ("rows_after_dedup", rows_after_dedup),
    ("missing_distance_km_before", missing_distance_before),
    ("missing_parcel_weight_before", missing_parcel_weight_before),
    ("sentinel_weight_rows", sentinel_weight_rows),
    ("missing_declared_value_before", missing_declared_value_before),
    ("missing_payment_mode_before", missing_payment_mode_before),
    ("service_type_spellings_before", service_type_spellings_before),
    ("service_type_spellings_after", service_type_spellings_after),
    ("median_distance_km", medians["distance_km"]),
    ("median_parcel_weight_kg", medians["parcel_weight_kg"]),
    ("median_declared_value_inr", medians["declared_value_inr"]),
    ("rows_after_hub_merge", rows_after_hub_merge),
    ("nulls_in_features_and_target", nulls_in_features_and_target),
    ("delayed_count", int(full["is_delayed"].sum())),
    ("delay_rate", float(full["is_delayed"].mean())),
    ("hours_q1", q1), ("hours_q3", q3),
    ("hours_lower_bound", lower_bound), ("hours_upper_bound", upper_bound),
    ("hours_outlier_count", hours_outlier_count),
    ("declared_value_mean_inr", float(full["declared_value_inr"].mean())),
    ("declared_value_median_inr", float(full["declared_value_inr"].median())),
    ("log_declared_value_mean", float(filled_log.mean())),
    ("log_declared_value_median", float(filled_log.median())),
]
pd.DataFrame(checks, columns=["metric", "value"]).to_csv("outputs/part_a_checks.csv", index=False, float_format="%.6f")

# Group delay evidence.
rows = []
for dimension in ["service_type", "hub_tier"]:
    for group, g in full.groupby(dimension, sort=True):
        rows.append({"dimension": dimension, "group": group, "shipments": len(g),
                     "delayed": int(g["is_delayed"].sum()), "delay_rate": g["is_delayed"].mean(),
                     "mean_hours": g["delivery_hours"].mean()})
full["distance_band"] = pd.cut(
    full["distance_km"], bins=[0, 200, 400, 600, 2000],
    labels=["0-200", "201-400", "401-600", "601+"], right=True, include_lowest=True
)
for group, g in full.groupby("distance_band", observed=False, sort=True):
    if len(g):
        rows.append({"dimension": "distance_band", "group": str(group), "shipments": len(g),
                     "delayed": int(g["is_delayed"].sum()), "delay_rate": g["is_delayed"].mean(),
                     "mean_hours": g["delivery_hours"].mean()})
pd.DataFrame(rows, columns=["dimension", "group", "shipments", "delayed", "delay_rate", "mean_hours"]).to_csv(
    "outputs/part_a_delay_by_group.csv", index=False, float_format="%.6f")

full.select_dtypes(include=[np.number]).corr().to_csv("outputs/part_a_correlations.csv", index_label="column", float_format="%.6f")

if rows_after_hub_merge != rows_after_dedup:
    raise AssertionError("Hub merge changed the deduplicated shipment row count")
if nulls_in_features_and_target != 0:
    raise AssertionError("Missing values remain in the modelling features or target")
print("Part A complete: data/modelling_table.csv and Part A outputs written.")
print("Cleaning medians:", medians)

Part A complete: data/modelling_table.csv and Part A outputs written.
Cleaning medians: {'distance_km': 314.95000000000005, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}


In [127]:
# Check which modelling columns contain missing values

print("Available columns:")
print(full.columns.tolist())

print("\nMissing values in modelling features:")
for col in FEATURES:
    if col in full.columns:
        print(f"{col}: {full[col].isna().sum()}")
    else:
        print(f"{col}: COLUMN NOT FOUND")

print("\nMissing values in target columns:")
for col in ["delivery_hours", "is_delayed"]:
    if col in full.columns:
        print(f"{col}: {full[col].isna().sum()}")

print("\nRows containing missing values:")
print(full[full.isna().any(axis=1)].head(10))

Available columns:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours', 'declared_value_inr', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity', 'promised_hours', 'is_delayed', 'distance_band']

Missing values in modelling features:
distance_km: 0
parcel_weight_kg: 0
declared_value_inr: 1200
payment_mode: 0
service_type: 0
hub_tier: 0

Missing values in target columns:
delivery_hours: 0
is_delayed: 0

Rows containing missing values:
  shipment_id hub_id service_type payment_mode  parcel_weight_kg  distance_km  \
0    SHP10001  HUB10      Express          COD              0.62        398.3   
1    SHP10002  HUB10     Standard          COD              4.25        312.2   
2    SHP10003  HUB21      Economy      Prepaid              0.81        421.3   
3    SHP10004  HUB03

In [128]:
import pandas as pd

raw_check = pd.read_csv("data/raw_shipments.csv")

print("Raw CSV columns:")
print(raw_check.columns.tolist())

print("\nDeclared value column checks:")

for col in ["declared_value", "declared_value_inr"]:
    if col in raw_check.columns:
        print(f"\n{col}")
        print("Non-missing:", raw_check[col].notna().sum())
        print("Missing:", raw_check[col].isna().sum())
        print("Sample values:")
        print(raw_check[col].head(10).tolist())
    else:
        print(f"\n{col}: NOT FOUND")

Raw CSV columns:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours']

Declared value column checks:

declared_value
Non-missing: 1212
Missing: 12
Sample values:
['₹1,290', '₹1,030', '₹441', '₹3,122', '₹276', '₹767', '₹2,698', '₹2,411', '₹679', '₹893']

declared_value_inr: NOT FOUND


In [129]:
from pathlib import Path

module_path = Path("src/shipment_prep.py")
source = module_path.read_text()

start = source.index("def clean_shipments(")
end = source.index("def build_features(", start)

new_function = '''
def clean_shipments(raw, medians=None):
    """Clean raw shipment data and impute missing feature values."""
    df = raw.drop_duplicates().copy()

    # Standardize categorical columns
    df["service_type"] = (
        df["service_type"].astype(str).str.strip().str.title()
    )
    df["payment_mode"] = df["payment_mode"].fillna("Unknown")

    # Convert the original rupee-formatted declared_value column
    df["declared_value_inr"] = (
        df["declared_value"]
        .astype("string")
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df["declared_value_inr"] = pd.to_numeric(
        df["declared_value_inr"], errors="coerce"
    )

    # Convert numeric columns
    for col in ["distance_km", "parcel_weight_kg"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Treat invalid sentinel weights as missing
    df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    # Calculate imputation medians from valid data
    if medians is None:
        medians = {
            "distance_km": float(df["distance_km"].median()),
            "parcel_weight_kg": float(df["parcel_weight_kg"].median()),
            "declared_value_inr": float(df["declared_value_inr"].median()),
        }

    # Fill missing feature values
    for col, value in medians.items():
        if pd.isna(value):
            raise ValueError(
                f"Cannot impute {col}: no valid values available."
            )
        df[col] = df[col].fillna(value)

    return df, medians


'''

source = source[:start] + new_function + source[end:]
module_path.write_text(source)

print("Updated clean_shipments() successfully.")

Updated clean_shipments() successfully.


In [130]:
import importlib
import src.shipment_prep

importlib.reload(src.shipment_prep)

from src.shipment_prep import clean_shipments, build_features, FEATURES

print("Module reloaded successfully!")

Module reloaded successfully!


In [152]:
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

clean_df, medians = clean_shipments(raw, medians=None)

print("Cleaning medians:")
print(medians)

print("\nMissing declared values after cleaning:")
print(clean_df["declared_value_inr"].isna().sum())

print("\nTotal rows after cleaning:", len(clean_df))

Cleaning medians:
{'distance_km': 314.95000000000005, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}

Missing declared values after cleaning:
0

Total rows after cleaning: 1200


In [132]:
from pathlib import Path

module_path = Path("src/shipment_prep.py")
source = module_path.read_text()

start = source.index("def clean_shipments(")
end = source.index("def build_features(", start)

new_function = '''
def clean_shipments(raw, medians=None):
    """Clean raw shipment data and impute missing feature values."""

    df = raw.drop_duplicates().copy()

    df["service_type"] = (
        df["service_type"].astype(str).str.strip().str.title()
    )

    df["payment_mode"] = df["payment_mode"].fillna("Unknown")

    # Convert declared values such as ₹1,290 into numeric values
    df["declared_value_inr"] = (
        df["declared_value"]
        .astype("string")
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    df["declared_value_inr"] = pd.to_numeric(
        df["declared_value_inr"], errors="coerce"
    ).astype("float64")

    # Ensure numeric columns support decimal median values
    for col in ["distance_km", "parcel_weight_kg"]:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        ).astype("float64")

    df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    if medians is None:
        medians = {
            "distance_km": float(df["distance_km"].median()),
            "parcel_weight_kg": float(df["parcel_weight_kg"].median()),
            "declared_value_inr": float(df["declared_value_inr"].median()),
        }

    for col, value in medians.items():
        if pd.isna(value):
            raise ValueError(
                f"Cannot impute {col}: no valid values available."
            )

        df[col] = df[col].astype("float64").fillna(float(value))

    return df, medians
'''

source = source[:start] + new_function + source[end:]
module_path.write_text(source)

print("clean_shipments() updated successfully.")

clean_shipments() updated successfully.


In [133]:
import importlib
import src.shipment_prep

importlib.reload(src.shipment_prep)

from src.shipment_prep import (
    clean_shipments,
    build_features,
    FEATURES
)

print("Module reloaded successfully.")

Module reloaded successfully.


In [134]:
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

clean_df, medians = clean_shipments(raw, medians=None)

print("Cleaning medians:")
print(medians)

print("\nMissing declared values:")
print(clean_df["declared_value_inr"].isna().sum())

print("\nCleaned dataset shape:")
print(clean_df.shape)

Cleaning medians:
{'distance_km': 314.95000000000005, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}

Missing declared values:
0

Cleaned dataset shape:
(1200, 18)


In [178]:
clean_full = build_features(clean_df, hubs)

print("Missing values in modelling features:")
print(clean_full[FEATURES].isna().sum())

print("\nMissing delivery_hours:")
print(clean_full["delivery_hours"].isna().sum())

print("\nMissing is_delayed:")
print(clean_full["is_delayed"].isna().sum())

print("\nFinal dataset shape:")
print(clean_full.shape)

Missing values in modelling features:
distance_km           0
parcel_weight_kg      0
declared_value_inr    0
payment_mode          0
service_type          0
hub_tier              0
dtype: int64

Missing delivery_hours:
0

Missing is_delayed:
0

Final dataset shape:
(1200, 24)


In [174]:
print(clean_full.columns.tolist())

['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours', 'declared_value_inr', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity', 'promised_hours']


In [183]:
threshold = 72

print("Threshold defined:", threshold)

Threshold defined: 72


In [184]:
clean_full['is_delayed'] = (clean_full['delivery_hours'] > threshold).astype(int)

In [186]:
print("Missing is_delayed values:")
print(clean_full["is_delayed"].isna().sum())

print("\nDelay distribution:")
print(clean_full["is_delayed"].value_counts())

print("\nDataset shape:")
print(clean_full.shape)

Missing is_delayed values:
0

Delay distribution:
is_delayed
0    895
1    305
Name: count, dtype: int64

Dataset shape:
(1200, 24)


In [190]:
print("Missing values in modelling features:")
print(clean_full[FEATURES].isna().sum())

print("\nMissing delivery_hours:")
print(clean_full["delivery_hours"].isna().sum())

print("\nMissing is_delayed:")
print(clean_full["is_delayed"].isna().sum())

print("\nFinal dataset shape:")
print(clean_full.shape)

Missing values in modelling features:
distance_km           0
parcel_weight_kg      0
declared_value_inr    0
payment_mode          0
service_type          0
hub_tier              0
dtype: int64

Missing delivery_hours:
0

Missing is_delayed:
0

Final dataset shape:
(1200, 24)


In [136]:
print("Current columns:")
print(clean_full.columns.tolist())

print("\nService types:")
print(clean_full["service_type"].value_counts())

Current columns:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours', 'declared_value_inr', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity', 'promised_hours']

Service types:
service_type
Standard    642
Economy     315
Express     243
Name: count, dtype: int64


In [137]:
PROMISED_HOURS = {
    "Express": 10,
    "Standard": 28,
    "Economy": 58
}

# Create promised delivery time
clean_full["promised_hours"] = (
    clean_full["service_type"].map(PROMISED_HOURS)
)

# Create delay target: 1 = delayed, 0 = not delayed
clean_full["is_delayed"] = (
    clean_full["delivery_hours"] > clean_full["promised_hours"]
).astype(int)

print("Target columns created successfully.")

Target columns created successfully.


In [138]:
print("Missing promised_hours:",
      clean_full["promised_hours"].isna().sum())

print("Missing is_delayed:",
      clean_full["is_delayed"].isna().sum())

print("\nDelay distribution:")
print(clean_full["is_delayed"].value_counts())

print("\nFinal dataset shape:")
print(clean_full.shape)

Missing promised_hours: 0
Missing is_delayed: 0

Delay distribution:
is_delayed
1    1200
Name: count, dtype: int64

Final dataset shape:
(1200, 24)


In [139]:
print("Feature missing values:")
print(clean_full[FEATURES].isna().sum())

print("\nTarget missing values:")
print(clean_full[["delivery_hours", "is_delayed"]].isna().sum())

Feature missing values:
distance_km           0
parcel_weight_kg      0
declared_value_inr    0
payment_mode          0
service_type          0
hub_tier              0
dtype: int64

Target missing values:
delivery_hours    0
is_delayed        0
dtype: int64


In [140]:
print(
    clean_full.groupby("service_type")["delivery_hours"]
    .agg(["count", "min", "max", "mean", "median"])
)

print("\nPromised hours by service type:")
print(clean_full.groupby("service_type")["promised_hours"].first())

print("\nDelay counts by service type:")
print(
    clean_full.groupby("service_type")["is_delayed"]
    .value_counts()
    .unstack(fill_value=0)
)

              count    min     max       mean  median
service_type                                         
Economy         315  63.17  117.18  82.093111   81.44
Express         243  16.36   75.93  36.805638   35.73
Standard        642  30.92   87.66  52.256916   51.79

Promised hours by service type:
service_type
Economy     58
Express     10
Standard    28
Name: promised_hours, dtype: int64

Delay counts by service type:
is_delayed      1
service_type     
Economy       315
Express       243
Standard      642


In [141]:
# Correct SLA thresholds
PROMISED_HOURS = {
    "Express": 36,
    "Standard": 60,
    "Economy": 96
}

# Recalculate promised delivery hours
clean_full["promised_hours"] = (
    clean_full["service_type"].map(PROMISED_HOURS)
)

# Recalculate delay target
clean_full["is_delayed"] = (
    clean_full["delivery_hours"] > clean_full["promised_hours"]
).astype(int)

print("Delay distribution:")
print(clean_full["is_delayed"].value_counts())

print("\nDelay percentage:")
print(
    (clean_full["is_delayed"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nDelay counts by service type:")
print(
    clean_full.groupby("service_type")["is_delayed"]
    .value_counts()
    .unstack(fill_value=0)
)

Delay distribution:
is_delayed
0    922
1    278
Name: count, dtype: int64

Delay percentage:
is_delayed
0    76.83
1    23.17
Name: proportion, dtype: float64

Delay counts by service type:
is_delayed      0    1
service_type          
Economy       288   27
Express       124  119
Standard      510  132


In [142]:
from pathlib import Path

path = Path("part_a_clean_eda.py")

if path.exists():
    source = path.read_text()

    old = '''PROMISED_HOURS = {
    "Express": 10,
    "Standard": 28,
    "Economy": 58
}'''

    new = '''PROMISED_HOURS = {
    "Express": 36,
    "Standard": 60,
    "Economy": 96
}'''

    if old in source:
        source = source.replace(old, new)
        path.write_text(source)
        print("Part A SLA thresholds updated successfully.")
    elif new in source:
        print("Part A already has the correct SLA thresholds.")
    else:
        print("Could not find the exact threshold block. Do not rerun Part A yet.")
else:
    print("part_a_clean_eda.py not found in the current directory.")

Could not find the exact threshold block. Do not rerun Part A yet.


In [143]:
import os
from pathlib import Path

print("Current directory:")
print(os.getcwd())

print("\nSearching for Part A script...")

matches = list(Path("/content").rglob("part_a_clean_eda.py"))

if matches:
    for file in matches:
        print("FOUND:", file)
else:
    print("Part A script was not found inside /content.")

Current directory:
/content

Searching for Part A script...
FOUND: /content/part_a_clean_eda.py


In [144]:
import os

print("Current directory:", os.getcwd())
print("\nFiles and folders inside /content:")

for item in os.listdir("/content"):
    print(item)

Current directory: /content

Files and folders inside /content:
.config
part_d_segments.py
part_a_clean_eda.py
part_b_regression.py
package_model.py
data
reports
src
outputs
models
predict_delay.py
part_c_classification.py
sample_data


In [145]:
from pathlib import Path

project_files = [
    "shipment_prep.py",
    "part_a_clean_eda.py",
    "part_b_regression.py",
    "part_c_classification.py",
    "part_d_segments.py",
    "package_model.py",
    "predict_delay.py"
]

for filename in project_files:
    matches = list(Path("/content").rglob(filename))

    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("MISSING:", filename)

FOUND: /content/src/shipment_prep.py
FOUND: /content/part_a_clean_eda.py
FOUND: /content/part_b_regression.py
FOUND: /content/part_c_classification.py
FOUND: /content/part_d_segments.py
FOUND: /content/package_model.py
FOUND: /content/predict_delay.py


In [146]:
import os

print("Current directory:", os.getcwd())

print("\nFiles in /content:")
for item in os.listdir("/content"):
    print(item)

Current directory: /content

Files in /content:
.config
part_d_segments.py
part_a_clean_eda.py
part_b_regression.py
package_model.py
data
reports
src
outputs
models
predict_delay.py
part_c_classification.py
sample_data


In [147]:
import os

# Create required folders
os.makedirs("/content/src", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/reports", exist_ok=True)

# Required Python files
files = [
    "part_a_clean_eda.py",
    "part_b_regression.py",
    "part_c_classification.py",
    "part_d_segments.py",
    "package_model.py",
    "predict_delay.py"
]

# Create missing files
for filename in files:
    filepath = f"/content/{filename}"

    if not os.path.exists(filepath):
        with open(filepath, "w") as f:
            f.write("# Shipment Delay Prediction Project\n")

        print("CREATED:", filepath)
    else:
        print("ALREADY EXISTS:", filepath)

print("\nFile creation completed!")

ALREADY EXISTS: /content/part_a_clean_eda.py
ALREADY EXISTS: /content/part_b_regression.py
ALREADY EXISTS: /content/part_c_classification.py
ALREADY EXISTS: /content/part_d_segments.py
ALREADY EXISTS: /content/package_model.py
ALREADY EXISTS: /content/predict_delay.py

File creation completed!


In [148]:
import os

required_files = [
    "src/shipment_prep.py",
    "part_a_clean_eda.py",
    "part_b_regression.py",
    "part_c_classification.py",
    "part_d_segments.py",
    "package_model.py",
    "predict_delay.py"
]

for file in required_files:
    path = f"/content/{file}"

    if os.path.exists(path):
        print("FOUND:", path)
    else:
        print("MISSING:", path)

FOUND: /content/src/shipment_prep.py
FOUND: /content/part_a_clean_eda.py
FOUND: /content/part_b_regression.py
FOUND: /content/part_c_classification.py
FOUND: /content/part_d_segments.py
FOUND: /content/package_model.py
FOUND: /content/predict_delay.py


In [149]:
import os

init_file = "/content/src/__init__.py"

if not os.path.exists(init_file):
    open(init_file, "w").close()

import sys
sys.path.insert(0, "/content")

import src.shipment_prep

print("Shipment preparation module imported successfully!")

Shipment preparation module imported successfully!


In [150]:
from pathlib import Path

print("ZIP files:")
for file in Path("/content").rglob("*.zip"):
    print(file)

print("\nPython files:")
for file in Path("/content").rglob("*.py"):
    print(file)

print("\nCSV files:")
for file in Path("/content").rglob("*.csv"):
    print(file)

ZIP files:

Python files:
/content/part_d_segments.py
/content/part_a_clean_eda.py
/content/part_b_regression.py
/content/package_model.py
/content/predict_delay.py
/content/part_c_classification.py
/content/src/__init__.py
/content/src/shipment_prep.py

CSV files:
/content/data/new_shipments.csv
/content/data/raw_shipments.csv
/content/data/hubs.csv
/content/data/modelling_table.csv
/content/outputs/part_a_delay_by_group.csv
/content/outputs/part_a_correlations.csv
/content/outputs/part_a_checks.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_test.csv


**part_b_regression.py**

In [204]:

"""Part B: Leakage-safe regression experiments for delivery_hours."""

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.shipment_prep import FEATURES

# Create output directory
os.makedirs("outputs", exist_ok=True)

# Load dataset
df = pd.read_csv("data/modelling_table.csv")

# Separate features and target
X = df[FEATURES].copy()
y = pd.to_numeric(df["delivery_hours"], errors="coerce")

# Remove rows with missing target
valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Preprocessing checks
check_preprocessor = clone(preprocessor)

train_encoded = check_preprocessor.fit_transform(X_train)
test_encoded = check_preprocessor.transform(X_test)

checks = [
    ("train_rows", len(X_train)),
    ("val_rows", len(X_val)),
    ("test_rows", len(X_test)),
    ("train_encoded_features", train_encoded.shape[1]),
    ("test_encoded_features", test_encoded.shape[1])
]

pd.DataFrame(
    checks, columns=["metric", "value"]
).to_csv("outputs/part_b_checks.csv", index=False)


# Helper function to create model pipelines
def make_pipeline(model):
    return Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])


# Tree-depth diagnostic
depths = [1, 2, 3, 4, 6, 8, 12, None]

tree_rows = []

for depth in depths:

    tree = make_pipeline(
        DecisionTreeRegressor(
            max_depth=depth,
            random_state=42
        )
    )

    tree.fit(X_train, y_train)

    train_pred = tree.predict(X_train)
    val_pred = tree.predict(X_val)

    tree_rows.append({
        "max_depth": "unlimited" if depth is None else depth,
        "train_rmse": np.sqrt(
            mean_squared_error(y_train, train_pred)
        ),
        "val_rmse": np.sqrt(
            mean_squared_error(y_val, val_pred)
        )
    })

tree_table = pd.DataFrame(tree_rows)

tree_table.to_csv(
    "outputs/part_b_tree_depth.csv",
    index=False,
    float_format="%.6f"
)

tree_table["depth_order"] = tree_table["max_depth"].map(
    lambda value: 999 if str(value) == "unlimited" else int(value)
)

best_depth_row = tree_table.sort_values(
    ["val_rmse", "depth_order"]
).iloc[0]

best_depth = (
    None if str(best_depth_row["max_depth"]) == "unlimited"
    else int(best_depth_row["max_depth"])
)

tree_table = tree_table.drop(columns=["depth_order"])


# Regression models
models = {
    "linear_regression": make_pipeline(
        LinearRegression()
    ),

    "decision_tree": make_pipeline(
        DecisionTreeRegressor(
            max_depth=best_depth,
            random_state=42
        )
    ),

    "random_forest": make_pipeline(
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    ),

    "gradient_boosting": make_pipeline(
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )
    )
}


# Evaluation function
def metric_row(name, split, truth, pred):

    mse = mean_squared_error(truth, pred)

    return {
        "model": name,
        "split": split,
        "MAE": mean_absolute_error(truth, pred),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "R2": r2_score(truth, pred)
    }


# Mean baseline
baseline_pred = np.full(
    len(y_val),
    float(y_train.mean())
)

metric_rows = [
    metric_row("mean_baseline", "val", y_val, baseline_pred)
]

fitted = {}

# Train and evaluate models
for name, model in models.items():

    model.fit(X_train, y_train)

    fitted[name] = model

    val_pred = model.predict(X_val)

    metric_rows.append(
        metric_row(name, "val", y_val, val_pred)
    )


# Learning-rate experiment
gb_rows = []

for lr in [0.01, 0.1, 1.0]:

    gb = make_pipeline(
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=lr,
            random_state=42
        )
    )

    gb.fit(X_train, y_train)

    pred = gb.predict(X_val)

    gb_rows.append({
        "learning_rate": lr,
        "val_rmse": np.sqrt(
            mean_squared_error(y_val, pred)
        ),
        "val_r2": r2_score(y_val, pred)
    })

pd.DataFrame(gb_rows).to_csv(
    "outputs/part_b_gb_learning_rate.csv",
    index=False,
    float_format="%.6f"
)


# Rank models using validation RMSE
ranked = sorted(
    models,
    key=lambda name: mean_squared_error(
        y_val,
        fitted[name].predict(X_val)
    ) ** 0.5
)

# Train + validation data
X_tv = pd.concat([X_train, X_val], axis=0)
y_tv = pd.concat([y_train, y_val], axis=0)

# Five-fold cross-validation
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_rows = []

for name in ranked[:2]:

    scores = cross_val_score(
        models[name],
        X_tv,
        y_tv,
        cv=kfold,
        scoring="r2"
    )

    cv_rows.append({
        "model": name,
        "mean_r2": scores.mean(),
        "std_r2": scores.std()
    })

pd.DataFrame(cv_rows).to_csv(
    "outputs/part_b_cv.csv",
    index=False,
    float_format="%.6f"
)


# Evaluate the validation winner on test data only once
final_name = ranked[0]

final_test_pred = fitted[final_name].predict(X_test)

metric_rows.append(
    metric_row(
        final_name,
        "test",
        y_test,
        final_test_pred
    )
)

# Save metrics
pd.DataFrame(
    metric_rows,
    columns=["model", "split", "MAE", "MSE", "RMSE", "R2"]
).to_csv(
    "outputs/part_b_metrics.csv",
    index=False,
    float_format="%.6f"
)

print("\nPart B completed successfully!")
print("Decision-tree depth:", best_depth)
print("Final candidate:", final_name)
print("Encoded feature count:", train_encoded.shape[1])
print("\nValidation results:")

print(
    pd.DataFrame(metric_rows)
    .query("split == 'val'")
    .sort_values("RMSE")
)


Numerical features: ['distance_km', 'parcel_weight_kg', 'declared_value_inr', 'hub_tier']
Categorical features: ['payment_mode', 'service_type']

Part B completed successfully!
Decision-tree depth: 4
Final candidate: linear_regression
Encoded feature count: 10

Validation results:
               model split        MAE         MSE       RMSE        R2
1  linear_regression   val   5.438243   49.149057   7.010639  0.858821
4  gradient_boosting   val   5.716680   55.159499   7.426944  0.841556
2      decision_tree   val   5.809765   56.395288   7.509680  0.838006
3      random_forest   val   5.872182   59.171595   7.692308  0.830032
0      mean_baseline   val  15.795189  355.520424  18.855249 -0.021221


**part_c_classification.py**

In [213]:
"""Part C: delay classification, feature evidence, tuning and alert threshold."""
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

# --- defensive import: NOISE_COLUMNS may not exist in src.shipment_prep ---
from src.shipment_prep import FEATURES
try:
    from src.shipment_prep import NOISE_COLUMNS
except ImportError:
    NOISE_COLUMNS = ()
# ---------------------------------------------------------------------------

os.makedirs("outputs", exist_ok=True)

# Load modelling dataset
df = pd.read_csv("data/modelling_table.csv")

# Separate features and target
X_raw = df[FEATURES].copy()
y = df["is_delayed"].astype(int)

# Identify categorical columns
categorical_cols = X_raw.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Categorical columns:", categorical_cols)

# Convert categorical features into numeric dummy variables
X = pd.get_dummies(
    X_raw,
    columns=categorical_cols,
    drop_first=False,
    dtype=float
)

# Ensure all columns are numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Handle missing values
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

# Update FEATURES to match the encoded dataset
FEATURES = X.columns.tolist()

# Final numeric conversion
X = X.astype(float)

print("\nEncoded feature count:", len(FEATURES))
print("Final X shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst five encoded rows:")
display(X.head())

print("\nRemaining missing values:", X.isna().sum().sum())

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.5, random_state=42, stratify=y_tv)


def metrics(name, truth, pred):
    tn, fp, fn, tp = confusion_matrix(truth, pred, labels=[0, 1]).ravel()
    return {"model": name, "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
            "accuracy": float(accuracy_score(truth, pred)),
            "precision": float(precision_score(truth, pred, zero_division=0)),
            "recall": float(recall_score(truth, pred, zero_division=0)),
            "f1": float(f1_score(truth, pred, zero_division=0))}


# Baseline and three validation classifiers.
model_rows = [metrics("blind_all_on_time", y_val,
                      np.zeros(len(y_val), dtype=int))]

log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
]).fit(X_train, y_train)
model_rows.append(metrics("logistic_regression", y_val, log_model.predict(X_val)))

tree_candidates = []
for depth in [2, 3, 4, 5, 6, 8, 10]:
    tree = DecisionTreeClassifier(
        criterion="gini", max_depth=depth, random_state=42).fit(X_train, y_train)
    pred = tree.predict(X_val)
    score = f1_score(y_val, pred, zero_division=0)
    tree_candidates.append((float(score), -depth, depth, tree, pred))

if tree_candidates:
    _, _, best_tree_depth, best_tree, best_tree_pred = max(
        tree_candidates, key=lambda x: (x[0], x[1]))
    model_rows.append(metrics(f"decision_tree_depth_{best_tree_depth}",
                              y_val, best_tree_pred))

rf_default_17 = RandomForestClassifier(n_estimators=100, random_state=42).fit(
    X_train, y_train)
model_rows.append(metrics("random_forest_default", y_val,
                          rf_default_17.predict(X_val)))

pd.DataFrame(model_rows).to_csv(
    "outputs/part_c_models.csv", index=False, float_format="%.6f")

# Feature F-scores and forest importance are fit on training rows only.
forest_for_importance = RandomForestClassifier(
    n_estimators=100, random_state=42).fit(X_train, y_train)
selector_all = SelectKBest(f_classif, k="all").fit(X_train, y_train)
scores = np.asarray(selector_all.scores_, dtype=float)
scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

order = np.argsort(-scores, kind="stable")
ranks = np.empty(len(scores), dtype=int)
ranks[order] = np.arange(1, len(scores) + 1)

feature_table = pd.DataFrame({
    "feature": FEATURES,
    "f_score": scores,
    "f_rank": ranks,
    "rf_importance": forest_for_importance.feature_importances_,
    "is_noise": [f in NOISE_COLUMNS for f in FEATURES],
})
feature_table.to_csv(
    "outputs/part_c_feature_scores.csv", index=False, float_format="%.6f")

# Validation F1 across the requested selector sizes.
selectk_rows = []
for k in [5, 7, 10, 12, 17]:
    k_eff = int(min(k, len(FEATURES)))
    pipe = Pipeline([
        ("selector", SelectKBest(f_classif, k=k_eff)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    selectk_rows.append({
        "k": k,
        "val_f1": float(f1_score(y_val, pipe.predict(X_val), zero_division=0)),
    })
pd.DataFrame(selectk_rows).to_csv(
    "outputs/part_c_selectk.csv", index=False, float_format="%.6f")

# Fit selector on training rows only; transform validation with the same selector.
K_SELECT = int(min(7, len(FEATURES)))
selector7 = SelectKBest(f_classif, k=K_SELECT).fit(X_train, y_train)
X_train7 = selector7.transform(X_train)
X_val7 = selector7.transform(X_val)

param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [4, 6, 8],
    "min_samples_leaf": [1, 5, 10],
}
n_candidates = (len(param_grid["n_estimators"])
                * len(param_grid["max_depth"])
                * len(param_grid["min_samples_leaf"]))
cv_folds = 5
# guard against tiny training sets where 5-fold CV is impossible
cv_folds = int(max(2, min(cv_folds, len(X_train7))))

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid=param_grid, cv=cv_folds)
grid.fit(X_train7, y_train)

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=int(min(9, n_candidates)),
    cv=cv_folds,
    random_state=42,
)
random_search.fit(X_train7, y_train)

deforest = RandomForestClassifier(
    n_estimators=100, random_state=42, max_depth=None, min_samples_leaf=1)
deforest.fit(X_train7, y_train)

tuning_rows = []
for name, candidates, folds, fits, params, best_acc, model in [
    ("grid_search", n_candidates, cv_folds, n_candidates * cv_folds,
     grid.best_params_, float(grid.best_score_), grid.best_estimator_),
    ("randomized_search", int(min(9, n_candidates)), cv_folds,
     int(min(9, n_candidates)) * cv_folds,
     random_search.best_params_, float(random_search.best_score_),
     random_search.best_estimator_),
    ("default_forest_same_7_features", 1, 0, 1,
     {"n_estimators": 100, "max_depth": "unlimited", "min_samples_leaf": 1},
     float("nan"), deforest),
]:
    tuning_rows.append({
        "search": name,
        "candidates": candidates,
        "cv_folds": folds,
        "total_fits": fits,
        "best_params": str(params),
        "best_cv_accuracy": best_acc,
        "val_f1": float(f1_score(y_val, model.predict(X_val7), zero_division=0)),
    })
pd.DataFrame(tuning_rows).to_csv(
    "outputs/part_c_tuning.csv", index=False, float_format="%.6f")

# Choose highest validation F1; ties: default, then grid, then randomized.
priority = {"default_forest_same_7_features": 0,
            "grid_search": 1,
            "randomized_search": 2}
tuning_df = pd.DataFrame(tuning_rows)
max_f1 = float(tuning_df["val_f1"].max())
chosen_tied = tuning_df[np.isclose(tuning_df["val_f1"].astype(float), max_f1,
                                   rtol=1e-9, atol=1e-12)]
chosen_name = min(chosen_tied["search"].tolist(),
                  key=lambda name: priority.get(name, 99))

chosen_params = {
    "grid_search": grid.best_params_,
    "randomized_search": random_search.best_params_,
    "default_forest_same_7_features": {"n_estimators": 100,
                                       "max_depth": None,
                                       "min_samples_leaf": 1},
}[chosen_name]

final_pipe = Pipeline([
    ("selector", SelectKBest(f_classif, k=K_SELECT)),
    ("model", RandomForestClassifier(random_state=42, **chosen_params)),
])
final_pipe.fit(X_train, y_train)

val_prob = final_pipe.predict_proba(X_val)[:, 1]
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_rows = []
for threshold in thresholds:
    m = metrics("threshold", y_val, (val_prob >= threshold).astype(int))
    row = {"threshold": threshold}
    row.update({k: v for k, v in m.items() if k != "model"})
    threshold_rows.append(row)
threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(
    "outputs/part_c_thresholds.csv", index=False, float_format="%.6f")

eligible = threshold_table[threshold_table["recall"] >= 0.80]
chosen_threshold = (float(eligible["threshold"].max())
                    if len(eligible) else 0.1)

# Test rows are evaluated once, after the final model and threshold are fixed.
test_prob = final_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)
test_metrics = metrics("final_test", y_test, test_pred)
pd.DataFrame([{
    "threshold": chosen_threshold,
    "TN": test_metrics["TN"],
    "FP": test_metrics["FP"],
    "FN": test_metrics["FN"],
    "TP": test_metrics["TP"],
    "accuracy": test_metrics["accuracy"],
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
}]).to_csv("outputs/part_c_final_test.csv", index=False, float_format="%.6f")

max_depth_value = chosen_params.get("max_depth", None)
pd.DataFrame([{
    "selector_k": K_SELECT,
    "n_estimators": chosen_params.get("n_estimators", 100),
    "max_depth": "unlimited" if max_depth_value is None else max_depth_value,
    "min_samples_leaf": chosen_params.get("min_samples_leaf", 1),
    "threshold": chosen_threshold,
    "chosen_from": chosen_name,
}]).to_csv("outputs/part_c_final_config.csv", index=False)

print("Part C complete. Selected:", chosen_name, "threshold:", chosen_threshold)

Categorical columns: ['payment_mode', 'service_type']

Encoded feature count: 10
Final X shape: (1200, 10)
Target shape: (1200,)

First five encoded rows:


,distance_km,parcel_weight_kg,declared_value_inr,hub_tier,payment_mode_COD,payment_mode_Prepaid,payment_mode_Unknown,service_type_Economy,service_type_Express,service_type_Standard
0,398.3,0.62,1290.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0
1,312.2,4.25,1030.0,2.0,1.0,0.0,0.0,0.0,0.0,1.0
2,421.3,0.81,441.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0
3,240.7,4.43,3122.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4,317.9,0.91,276.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0



Remaining missing values: 0
Part C complete. Selected: default_forest_same_7_features threshold: 0.6


In [210]:
import pandas as pd

df = pd.read_csv("data/modelling_table.csv")

print("Target column information:")
print(df["is_delayed"].value_counts(dropna=False))

print("\nTarget percentages:")
print(df["is_delayed"].value_counts(normalize=True, dropna=False) * 100)

print("\nUnique target values:")
print(df["is_delayed"].unique())

Target column information:
is_delayed
1    1200
Name: count, dtype: int64

Target percentages:
is_delayed
1    100.0
Name: proportion, dtype: float64

Unique target values:
[1]


In [211]:
import pandas as pd

df = pd.read_csv("data/modelling_table.csv")

# Use the actual delay threshold specified by your project
threshold = 72

df["is_delayed"] = (
    df["delivery_hours"] > threshold
).astype(int)

print("Corrected target distribution:")
print(df["is_delayed"].value_counts())

# Save corrected dataset
df.to_csv("data/modelling_table.csv", index=False)

Corrected target distribution:
is_delayed
0    895
1    305
Name: count, dtype: int64


In [212]:
df = pd.read_csv("data/modelling_table.csv")

assert df["is_delayed"].nunique() == 2, (
    "ERROR: Target must contain both 0 and 1. "
    "Check the delay threshold."
)

print("SUCCESS: Both target classes are available.")
print(df["is_delayed"].value_counts())

SUCCESS: Both target classes are available.
is_delayed
0    895
1    305
Name: count, dtype: int64


In [215]:
import pandas as pd

df = pd.read_csv("data/modelling_table.csv")

from src.shipment_prep import FEATURES

print("Features in your dataset:")
print(FEATURES)

print("\nData types:")
print(df[FEATURES].dtypes)

print("\nCategorical columns:")
print(df[FEATURES].select_dtypes(include=["object", "category"]).columns.tolist())

Features in your dataset:
['distance_km', 'parcel_weight_kg', 'declared_value_inr', 'payment_mode', 'service_type', 'hub_tier']

Data types:
distance_km           float64
parcel_weight_kg      float64
declared_value_inr    float64
payment_mode           object
service_type           object
hub_tier                int64
dtype: object

Categorical columns:
['payment_mode', 'service_type']


In [199]:
from src.shipment_prep import FEATURES

print("FEATURES:", FEATURES)

FEATURES: ['distance_km', 'parcel_weight_kg', 'declared_value_inr', 'payment_mode', 'service_type', 'hub_tier']


In [197]:
NOISE_COLUMNS = []

In [216]:
import os

expected_files = [
    "outputs/part_c_models.csv",
    "outputs/part_c_feature_scores.csv",
    "outputs/part_c_selectk.csv",
    "outputs/part_c_tuning.csv",
    "outputs/part_c_thresholds.csv",
    "outputs/part_c_final_test.csv",
    "outputs/part_c_final_config.csv"
]

for file in expected_files:
    if os.path.exists(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

FOUND: outputs/part_c_models.csv
FOUND: outputs/part_c_feature_scores.csv
FOUND: outputs/part_c_selectk.csv
FOUND: outputs/part_c_tuning.csv
FOUND: outputs/part_c_thresholds.csv
FOUND: outputs/part_c_final_test.csv
FOUND: outputs/part_c_final_config.csv


**part_d_segments.py**

In [217]:
"""Part D: hub profiles, fixed K=3 segments and tier mismatch evidence."""
import os
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

os.makedirs("outputs", exist_ok=True)
df = pd.read_csv("data/modelling_table.csv")
hubs = pd.read_csv("data/hubs.csv")
profile = (df.groupby("hub_id", sort=True)
           .agg(shipments=("is_delayed", "size"),
                mean_delivery_hours=("delivery_hours", "mean"),
                delay_rate=("is_delayed", "mean"),
                mean_distance_km=("distance_km", "mean"),
                mean_n_stops=("n_stops", "mean"))
           .reset_index().sort_values("hub_id").reset_index(drop=True))
profile.to_csv("outputs/hub_profile.csv", index=False, float_format="%.6f")

profile_columns = ["mean_delivery_hours", "delay_rate", "mean_distance_km", "mean_n_stops"]
X_scaled = StandardScaler().fit_transform(profile[profile_columns])


def across_over_within(X, labels):
    within, across = [], []
    for i, j in combinations(range(len(X)), 2):
        distance = float(np.linalg.norm(X[i] - X[j]))
        (within if labels[i] == labels[j] else across).append(distance)
    return float(np.mean(across) / np.mean(within))

separation_rows = []
for k in [2, 3, 4, 5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sizes = sorted(np.bincount(model.labels_).tolist(), reverse=True)
    separation_rows.append({"K": k, "across_over_within": across_over_within(X_scaled, model.labels_),
                            "cluster_sizes": "-".join(map(str, sizes))})
pd.DataFrame(separation_rows).to_csv("outputs/part_d_separation.csv", index=False, float_format="%.6f")

kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
profile["cluster"] = kmeans3.labels_
cluster_order = profile.groupby("cluster")["delay_rate"].mean().sort_values().index.tolist()
segment_names = ["Standard", "Watch", "Escalate"]
cluster_to_segment = dict(zip(cluster_order, segment_names))
profile["segment"] = profile["cluster"].map(cluster_to_segment)

hub_segments = profile[["hub_id", "segment"]].merge(
    hubs[["hub_id", "hub_city", "hub_tier"]], on="hub_id", how="left", validate="one_to_one")
hub_segments = hub_segments[["hub_id", "segment", "hub_city", "hub_tier"]]
hub_segments.to_csv("outputs/hub_segments.csv", index=False)

segment_profile = (profile.groupby("segment")
                   .agg(hubs=("hub_id", "size"),
                        mean_delivery_hours=("mean_delivery_hours", "mean"),
                        delay_rate=("delay_rate", "mean"),
                        mean_distance_km=("mean_distance_km", "mean"),
                        mean_n_stops=("mean_n_stops", "mean"))
                   .reindex(segment_names).reset_index())
segment_profile.to_csv("outputs/segment_profile.csv", index=False, float_format="%.6f")

tier_expected = {1: "Standard", 2: "Watch", 3: "Escalate"}
hub_segments["expected_segment"] = hub_segments["hub_tier"].map(tier_expected)
mismatch = hub_segments[hub_segments["segment"] != hub_segments["expected_segment"]]
mismatch[["hub_id", "segment", "hub_city", "hub_tier"]].to_csv(
    "outputs/tier_vs_segment_mismatch.csv", index=False)
print("Part D segmentation complete.")

Part D segmentation complete.


**package_model.py**

In [236]:
"""Rebuild Part C's final pipeline and persist the reusable model artefacts."""
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from src.shipment_prep import FEATURES, clean_shipments, build_features

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")
clean, medians = clean_shipments(raw)
full = build_features(clean, hubs)
full["is_delayed"] = (full["delivery_hours"] > full["promised_hours"]).astype(int)
X = full[FEATURES].astype(float)
y = full["is_delayed"].astype(int)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.5, random_state=42, stratify=y_tv)

cfg = pd.read_csv("outputs/part_c_final_config.csv").iloc[0]
max_depth_text = str(cfg["max_depth"])
max_depth = None if max_depth_text.lower() == "unlimited" else int(float(max_depth_text))
rf_params = {"n_estimators": int(cfg["n_estimators"]), "max_depth": max_depth,
             "min_samples_leaf": int(cfg["min_samples_leaf"]), "random_state": 42}

# This definition fixes the earlier NameError: final_pipe is created here before joblib.dump.
final_pipe = Pipeline([("selector", SelectKBest(f_classif, k=int(cfg["selector_k"]))),
                       ("model", RandomForestClassifier(**rf_params))])
final_pipe.fit(X_train, y_train)

joblib.dump(final_pipe, "models/delay_model.pkl")
pd.DataFrame({"threshold": [float(cfg["threshold"])]}).to_csv("models/delay_threshold.csv", index=False)
pd.DataFrame({"feature": FEATURES}).to_csv("models/feature_order.csv", index=False)
pd.DataFrame({"column": list(medians.keys()), "median": list(medians.values())}).to_csv(
    "models/impute_values.csv", index=False)

reloaded = joblib.load("models/delay_model.pkl")
in_memory_prob = final_pipe.predict_proba(X_test)[:, 1]
reloaded_prob = reloaded.predict_proba(X_test)[:, 1]
diff = float(np.max(np.abs(in_memory_prob - reloaded_prob)))
pd.DataFrame({"metric": ["test_rows_scored", "reload_max_abs_probability_diff", "threshold"],
              "value": [len(X_test), diff, float(cfg["threshold"])]}).to_csv(
    "outputs/part_d_reload_check.csv", index=False, float_format="%.12f")
print("Model packaged. final_pipe fitted; reload difference:", diff)

ValueError: could not convert string to float: 'COD'

In [222]:
from src.shipment_prep import FEATURES

print("Features used by the model:")
print(FEATURES)

print("\nCategorical columns:")
print(full[FEATURES].select_dtypes(include=["object", "category"]).columns.tolist())

print("\nData types:")
print(full[FEATURES].dtypes)

Features used by the model:
['distance_km', 'parcel_weight_kg', 'declared_value_inr', 'payment_mode', 'service_type', 'hub_tier']

Categorical columns:
['payment_mode', 'service_type']

Data types:
distance_km                  float64
parcel_weight_kg             float64
declared_value_inr           float64
payment_mode                  object
service_type                  object
hub_tier              string[python]
dtype: object


In [225]:
import os

required_files = [
    "models/delay_model.pkl",
    "models/delay_threshold.csv",
    "models/feature_order.csv",
    "models/impute_values.csv",
    "outputs/part_d_reload_check.csv"
]

for file in required_files:
    if os.path.exists(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

FOUND: models/delay_model.pkl
FOUND: models/delay_threshold.csv
FOUND: models/feature_order.csv
FOUND: models/impute_values.csv
MISSING: outputs/part_d_reload_check.csv


In [226]:
import pandas as pd
import numpy as np

print("Delivery hours summary:")
print(full["delivery_hours"].describe())

print("\nPromised hours summary:")
print(full["promised_hours"].describe())

print("\nTarget distribution:")
print(full["is_delayed"].value_counts(dropna=False))

print("\nTarget percentages:")
print(
    full["is_delayed"]
    .value_counts(normalize=True, dropna=False) * 100
)

Delivery hours summary:
count    1200.000000
mean       56.960033
std        19.203698
min        16.360000
25%        43.255000
50%        54.415000
75%        72.562500
max       117.180000
Name: delivery_hours, dtype: float64

Promised hours summary:
count    1200.000000
mean       32.230000
std        16.858349
min        10.000000
25%        28.000000
50%        28.000000
75%        58.000000
max        58.000000
Name: promised_hours, dtype: float64

Target distribution:
is_delayed
1    1200
Name: count, dtype: int64

Target percentages:
is_delayed
1    100.0
Name: proportion, dtype: float64


In [231]:
import pandas as pd
import numpy as np

# Recreate target using a 12-hour grace period
DELAY_GRACE_HOURS = 12

full["is_delayed"] = (
    full["delivery_hours"] >
    full["promised_hours"] + DELAY_GRACE_HOURS
).astype(int)

print("Target generated successfully!")

Target generated successfully!


In [238]:

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold

from src.shipment_prep import FEATURES, clean_shipments, build_features

# 1. Create folders
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# 2. Load data
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

clean, medians = clean_shipments(raw)
full = build_features(clean, hubs)

# 3. Create target with 12-hour grace period
DELAY_GRACE_HOURS = 12

full["is_delayed"] = (
    full["delivery_hours"] >
    full["promised_hours"] + DELAY_GRACE_HOURS
).astype(int)

# 4. Prepare features and target
X = full[FEATURES].copy()
y = full["is_delayed"].astype(int)

# 5. Split numerical and categorical columns
numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = [
    c for c in FEATURES if c not in numeric_cols
]

# 6. Split data
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv,
    test_size=0.50,
    random_state=42,
    stratify=y_tv
)

# 7. Load final configuration
cfg = pd.read_csv(
    "outputs/part_c_final_config.csv"
).iloc[0]

max_depth_text = str(cfg["max_depth"]).strip()

max_depth = (
    None if max_depth_text.lower()
    in ["unlimited", "none", "nan"]
    else int(float(max_depth_text))
)

rf_params = {
    "n_estimators": int(cfg["n_estimators"]),
    "max_depth": max_depth,
    "min_samples_leaf": int(cfg["min_samples_leaf"]),
    "random_state": 42,
    "n_jobs": -1
}

# 8. Numerical preprocessing
transformers = []

if numeric_cols:
    transformers.append((
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_cols
    ))

# 9. Categorical preprocessing
if categorical_cols:
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    transformers.append((
        "categorical",
        cat_pipeline,
        categorical_cols
    ))

preprocessor = ColumnTransformer(
    transformers=transformers
)

# 10. Determine encoded feature count
encoded = preprocessor.fit_transform(X_train)

variance_filter = VarianceThreshold(0.0)
encoded_filtered = variance_filter.fit_transform(encoded)

encoded_count = encoded_filtered.shape[1]

selector_k = min(
    int(cfg["selector_k"]),
    encoded_count
)

# 11. Create final pipeline
final_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(0.0)),
    ("selector", SelectKBest(f_classif, k=selector_k)),
    ("model", RandomForestClassifier(**rf_params))
])

# 12. Train model
final_pipe.fit(X_train, y_train)

print("Model training completed!")

# 13. Save model
joblib.dump(final_pipe, "models/delay_model.pkl")

pd.DataFrame({
    "threshold": [float(cfg["threshold"])]
}).to_csv("models/delay_threshold.csv", index=False)

pd.DataFrame({
    "feature": FEATURES
}).to_csv("models/feature_order.csv", index=False)

pd.DataFrame({
    "column": list(medians.keys()),
    "median": list(medians.values())
}).to_csv("models/impute_values.csv", index=False)

# 14. Reload model
reloaded = joblib.load("models/delay_model.pkl")

# 15. Calculate positive-class probability
classes = list(final_pipe.named_steps["model"].classes_)

positive_index = classes.index(1)

prob1 = final_pipe.predict_proba(X_test)[:, positive_index]
prob2 = reloaded.predict_proba(X_test)[:, positive_index]

diff = float(np.max(np.abs(prob1 - prob2)))

# 16. Save verification report
pd.DataFrame({
    "metric": [
        "test_rows_scored",
        "encoded_feature_count",
        "selected_feature_count",
        "reload_max_abs_probability_diff",
        "threshold"
    ],
    "value": [
        len(X_test),
        encoded.shape[1],
        selector_k,
        diff,
        float(cfg["threshold"])
    ]
}).to_csv(
    "outputs/part_d_reload_check.csv",
    index=False
)

# 17. Print results
print("\n========== PART C COMPLETE ==========")

print("Target distribution:")
print(y.value_counts())

print("Model classes:", classes)
print("Encoded features:", encoded.shape[1])
print("Selected features:", selector_k)
print("Test rows:", len(X_test))
print("Reload difference:", diff)

print("\nAll model artefacts saved successfully!")

assert diff < 1e-12


Model training completed!

========== PART C COMPLETE ==========
Target distribution:
is_delayed
1    1081
0     119
Name: count, dtype: int64
Model classes: [np.int64(0), np.int64(1)]
Encoded features: 12
Selected features: 7
Test rows: 360
Reload difference: 0.0

All model artefacts saved successfully!


In [233]:
import os

files = [
    "models/delay_model.pkl",
    "models/delay_threshold.csv",
    "models/feature_order.csv",
    "models/impute_values.csv",
    "outputs/part_d_reload_check.csv"
]

for f in files:
    print(
        "FOUND:" if os.path.exists(f) else "MISSING:",
        f
    )

FOUND: models/delay_model.pkl
FOUND: models/delay_threshold.csv
FOUND: models/feature_order.csv
FOUND: models/impute_values.csv
FOUND: outputs/part_d_reload_check.csv


In [220]:
import os

required_files = [
    "data/raw_shipments.csv",
    "data/hubs.csv",
    "src/shipment_prep.py",
    "outputs/part_c_final_config.csv"
]

for file in required_files:
    if os.path.exists(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

FOUND: data/raw_shipments.csv
FOUND: data/hubs.csv
FOUND: src/shipment_prep.py
FOUND: outputs/part_c_final_config.csv


**predict_delay.py**

In [255]:
"""Score new shipments and produce the deterministic operations action list."""
import pandas as pd
import joblib
from src.shipment_prep import clean_shipments, build_features

new_raw = pd.read_csv("data/new_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")
model = joblib.load("models/delay_model.pkl")
threshold = float(pd.read_csv("models/delay_threshold.csv")["threshold"].iloc[0])
feature_order = pd.read_csv("models/feature_order.csv")["feature"].tolist()
impute = pd.read_csv("models/impute_values.csv")
medians = dict(zip(impute["column"], impute["median"].astype(float)))
hub_segments = pd.read_csv("outputs/hub_segments.csv")

clean_new, _ = clean_shipments(new_raw, medians=medians)
features = build_features(clean_new, hubs)
missing = [column for column in feature_order if column not in features.columns]
if missing:
    raise RuntimeError(f"Missing model features: {missing}")
probabilities = model.predict_proba(features[feature_order].astype(float))[:, 1]

out = features[["shipment_id", "hub_id"]].copy()
out["delay_probability"] = [round(float(p), 4) for p in probabilities]
out["flagged"] = (probabilities >= threshold).astype(int)
out = out.merge(hub_segments[["hub_id", "segment"]], on="hub_id", how="left", validate="many_to_one")
if out["segment"].isna().any():
    raise ValueError("A new shipment hub has no matching hub segment")

def choose_action(row):
    if row["flagged"] == 1 and row["segment"] == "Escalate":
        return "Escalate to hub manager"
    if row["flagged"] == 1:
        return "Proactive customer call"
    return "No action"

out["action"] = out.apply(choose_action, axis=1)
out = out[["shipment_id", "hub_id", "segment", "delay_probability", "flagged", "action"]]
out.to_csv("outputs/new_shipment_actions.csv", index=False)
print("Scored rows:", len(out), "Flagged:", int(out["flagged"].sum()))
print(out["action"].value_counts().to_string())

ValueError: could not convert string to float: 'Prepaid'

In [240]:
import pandas as pd

new_raw = pd.read_csv("data/new_shipments.csv")

print("New shipment columns:")
print(new_raw.columns.tolist())

print("\nFirst five rows:")
display(new_raw.head())

New shipment columns:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no']

First five rows:


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no
0,SHP90001,HUB22,Standard,Prepaid,1.52,478.7,₹483,78,0,0,2026-08-03 17:26,9034,5,311,0.710,273
1,SHP90002,HUB24,Standard,COD,0.76,405.5,"₹1,137",94,1,0,2026-08-21 12:52,7336,8,163,0.784,284
2,SHP90003,HUB18,Standard,Prepaid,2.00,552.4,₹195,99,0,0,2026-08-07 16:29,9987,3,291,0.927,140
3,SHP90004,HUB12,Economy,Prepaid,1.66,243.0,"₹1,018",65,0,0,2026-08-28 14:41,4749,14,257,0.405,21
4,SHP90005,HUB22,Express,COD,0.72,219.1,NaN,72,0,0,2026-08-25 18:11,1364,11,442,0.832,164


In [251]:
from pathlib import Path

file_path = Path("/content/src/shipment_prep.py")

print(file_path.read_text())


import numpy as np
import pandas as pd

# SLA mapping from the Part A project implementation
PROMISED_HOURS = {
    "Express": 10,
    "Standard": 28,
    "Economy": 58
}

FEATURES = [
    "distance_km",
    "parcel_weight_kg",
    "declared_value_inr",
    "payment_mode",
    "service_type",
    "hub_tier"
]




def clean_shipments(raw, medians=None):
    """Clean raw shipment data and impute missing feature values."""

    df = raw.drop_duplicates().copy()

    df["service_type"] = (
        df["service_type"].astype(str).str.strip().str.title()
    )

    df["payment_mode"] = df["payment_mode"].fillna("Unknown")

    # Convert declared values such as ₹1,290 into numeric values
    df["declared_value_inr"] = (
        df["declared_value"]
        .astype("string")
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    df["declared_value_inr"] = pd.to_numeric(
        df["declared_value_inr"], errors="coerce"
    ).astype("float64")

    # En

In [252]:
from pathlib import Path

file_path = Path("/content/src/shipment_prep.py")
code = file_path.read_text()

start = code.index("def build_features(")

# Find the next function, if one exists
next_func = code.find("\ndef ", start + 1)

new_function = '''def build_features(clean, hubs):
    df = clean.copy()

    # Delivery hours is a target variable, not a prediction input.
    df = df.drop(columns=["delivery_hours"], errors="ignore")

    required = ["shipment_id", "hub_id", "distance_km",
                "parcel_weight_kg", "declared_value_inr",
                "payment_mode", "service_type"]

    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.merge(hubs, on="hub_id", how="left",
                  validate="many_to_one")

    df["hub_tier"] = df["hub_tier"].astype(float)

    return df
'''

if next_func == -1:
    code = code[:start] + new_function + "\n"
else:
    code = code[:start] + new_function + code[next_func:]

file_path.write_text(code)

print("build_features() updated successfully.")

build_features() updated successfully.


In [253]:
import importlib
import src.shipment_prep as shipment_prep

importlib.reload(shipment_prep)

clean_shipments = shipment_prep.clean_shipments
build_features = shipment_prep.build_features

print("Shipment preparation module reloaded.")

Shipment preparation module reloaded.


.

In [159]:
import os

print("Current directory:", os.getcwd())
print("\nFiles and folders:")

for item in os.listdir():
    print(item)

Current directory: /content

Files and folders:
.config
part_d_segments.py
part_a_clean_eda.py
part_b_regression.py
package_model.py
data
reports
src
outputs
models
predict_delay.py
part_c_classification.py
sample_data


In [160]:
import os

print("src folder exists:", os.path.isdir("src"))
print("shipment_prep.py exists:", os.path.isfile("src/shipment_prep.py"))

src folder exists: True
shipment_prep.py exists: True


In [161]:
import os

os.makedirs("src", exist_ok=True)

In [162]:
import shutil

if os.path.isfile("shipment_prep.py"):
    shutil.move("shipment_prep.py", "src/shipment_prep.py")

In [163]:
with open("src/__init__.py", "w") as f:
    pass

In [164]:
import os
import sys

PROJECT_DIR = os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("Python import path updated.")

Project directory: /content
Python import path updated.


In [165]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [166]:
!python part_a_clean_eda.py

In [167]:
!python part_b_regression.py

In [168]:
!python part_c_classification.py

In [169]:
!python part_d_segments.py

In [170]:
!python package_model.py

In [171]:
!python predict_delay.py

In [172]:
import os

required_files = [
    "src/__init__.py",
    "src/shipment_prep.py",
    "part_a_clean_eda.py",
    "part_b_regression.py",
    "part_c_classification.py",
    "part_d_segments.py",
    "package_model.py",
    "predict_delay.py"
]

for file in required_files:
    status = "FOUND" if os.path.isfile(file) else "MISSING"
    print(f"{file}: {status}")

src/__init__.py: FOUND
src/shipment_prep.py: FOUND
part_a_clean_eda.py: FOUND
part_b_regression.py: FOUND
part_c_classification.py: FOUND
part_d_segments.py: FOUND
package_model.py: FOUND
predict_delay.py: FOUND
